# 🎯 Notebook 3: Content Recommendation System
### Edu-Platform - Learning Style Based Recommender

**Goal:** Har student ke cluster ke hisaab se sahi content recommend karna - videos, notes, quizzes, assignments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle
import os
import warnings
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

os.makedirs('../plots', exist_ok=True)
os.makedirs('../../data', exist_ok=True)

## 📌 Step 1: Content Dataset

In [ ]:
content_data = [
    {"id": 1, "type": "video", "difficulty": 1, "subject": "Python", "tags": ["visual"]},
    {"id": 2, "type": "article", "difficulty": 2, "subject": "Python", "tags": ["theory"]},
    {"id": 3, "type": "quiz", "difficulty": 3, "subject": "Python", "tags": ["practice"]},
    {"id": 4, "type": "notes", "difficulty": 1, "subject": "Math", "tags": ["structured"]},
    {"id": 5, "type": "video", "difficulty": 2, "subject": "Math", "tags": ["visual"]}
]

df_content = pd.DataFrame(content_data)
df_content

## 📌 Step 2: Learning Style Rules

In [ ]:
LEARNING_STYLE_RULES = {
    'visual_learner': {
        'preferred_types': ['video'],
        'secondary_types': ['notes'],
        'avoid_types': ['article'],
        'tags_boost': ['visual'],
    },
    'theory_learner': {
        'preferred_types': ['article'],
        'secondary_types': ['notes'],
        'avoid_types': ['quiz'],
        'tags_boost': ['theory'],
    },
    'practical_learner': {
        'preferred_types': ['quiz'],
        'secondary_types': ['video'],
        'avoid_types': ['article'],
        'tags_boost': ['practice'],
    },
    'structured_learner': {
        'preferred_types': ['notes'],
        'secondary_types': ['article'],
        'avoid_types': [],
        'tags_boost': ['structured'],
    }
}

## 📌 Step 3: Scoring Function

In [ ]:
def score_content(content_row, learning_style, user_level=2):
    rules = LEARNING_STYLE_RULES[learning_style]
    score = 0

    if content_row['type'] in rules['preferred_types']:
        score += 5
    if content_row['type'] in rules['secondary_types']:
        score += 3
    if content_row['type'] in rules['avoid_types']:
        score -= 3

    for tag in content_row['tags']:
        if tag in rules['tags_boost']:
            score += 1

    diff_gap = abs(content_row['difficulty'] - user_level)
    score += max(0, 3 - diff_gap)

    return score

## 📌 Step 4: Recommendation Engine

In [ ]:
def recommend_content(learning_style, subject=None, user_level=2, top_n=5):
    df = df_content.copy()

    if subject:
        df = df[df['subject'] == subject]

    df['score'] = df.apply(lambda row: score_content(row, learning_style, user_level), axis=1)
    df = df.sort_values(by='score', ascending=False)

    return df.head(top_n)

## 📌 Step 5: Similarity-Based Recommendation

In [ ]:
features = pd.get_dummies(df_content[['type', 'difficulty']])
similarity_matrix = cosine_similarity(features)
similarity_df = pd.DataFrame(similarity_matrix)
similarity_df

In [ ]:
def recommend_similar(content_id):
    scores = similarity_df[content_id - 1]
    return df_content.iloc[scores.sort_values(ascending=False).index]

## 📌 Step 6: Testing

In [ ]:
print(recommend_content('visual_learner', subject='Python', user_level=1))
print(recommend_similar(1))

## 📌 Step 7: Save Model

In [ ]:
with open('../../data/content_recommender.pkl', 'wb') as f:
    pickle.dump(df_content, f)

## 📌 Step 8: API Helper

In [ ]:
def get_recommendations_api(style, subject, level):
    recs = recommend_content(style, subject, level)
    return recs.to_dict(orient='records')